# Stage 4 Manual Repair Walkthrough for `GOLDEN`

This notebook is a manual repair walkthrough for the Stage 4 checkpoint at `data/.private/GOLDEN/run/stage-4-checkpoints/review:prior_system`. The checkpoint is paused on a failing Stage 4 validation and the reducer's state machine cannot derive a concrete repair scope for the remaining diagnostics.

## Goal

Drive the checkpoint to a passing Stage 4 validation — `validate_assembly(...)` returning `is_valid=True` with **all** of:
- `compile_ok=True`,
- `pp_checked=True` and `pp_valid=True`,
- `sensitivity_consulted=True` and `sensitivity_valid=True`.

Both the prior predictive check and the sensitivity check must actually run and pass. A result where `pp_checked` or `sensitivity_consulted` is `False`, or where either gate reports `valid=False`, does not count as done. The state machine's routing / scope restrictions do **not** apply here. You are **not** limited to repair actions the router would emit, the scopes it can localize, or the parameters it currently considers "free". If a change to `model_spec` decisions, authored priors, or policy flags clears validation without tripping the hard invariants below, it is a legitimate fix for this notebook's purposes, even if the state machine would never have proposed it on its own.

Work iteratively: make a candidate change, re-run `validate_assembly`, inspect what still fails, adjust, and repeat until both PPC and sensitivity pass.

## Hard Invariants (Never Violate These)

These are not router conventions; they are structural guarantees the executable layer must preserve against `causal_spec`. A repair that breaks any of them is silently editing the model away from the declarative spec and must not be used, even if it would make validation pass.

### Structural freeze

No edits to:
- latent constructs,
- causal / estimation edges,
- measurement indicator assignments,
- invariance assumptions like whether `chronotype` is person-level invariant,
- estimation state membership.

### Forbidden parameter-surface moves

- removing entries from `model_spec["parameters"]` as a repair move,
- flipping any compiled `SSMSpec` mask to pin a parameter; this includes `drift_offdiag_mask`, `lambda_mask`, `manifest_means_mask`, `diffusion_chol_mask`, and `manifest_chol_diag_mask`,
- changing a locked observation distribution or link function on any indicator.

Pinning a parameter at its prior mean is a Dirac prior: the same as asserting the parameter's value by decree. For causal content (`beta_*` cross-lags, `tau_*` confounder factors) this is covert graph editing; for core SSM content (`sigma_*` diffusion) it changes what kind of process the latent is; for measurement scale (`lambda_*`) it commits to a measurement-invariance claim. None of these belong inside an executable-layer repair while `causal_spec` is frozen.

## What Is Fair Game

Everything else. In particular, you may freely adjust:
- prior distributions and hyperparameters on any parameter that currently exists in `model_spec["parameters"]`, regardless of whether the router would localize it,
- model-level policy flags the spec already exposes (initialization policy, observation intercept policy, equilibrium forcing, etc.),
- ambiguous-indicator distribution / link choices, subject to the compiler actually accepting them,
- the order and combination of these moves across multiple validation passes.

Success is defined by the validator, not by the reducer's scope heuristics. If the final `validate_assembly` call returns a valid result with both PPC and sensitivity checks consulted and passing, and no hard invariant above has been violated along the way, the repair is done.

In [1]:
from __future__ import annotations

import copy
import json
import math
import sys
from pathlib import Path
from pprint import pprint

import cloudpickle
import numpy as np
import polars as pl

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'apps').exists():
    for parent in REPO_ROOT.parents:
        if (parent / 'apps').exists():
            REPO_ROOT = parent
            break

sys.path.insert(0, str(REPO_ROOT / 'apps/data-pipeline/src'))

from causal_ssm_agent.flows.stages.stage4.agentic.stage4_orchestrator import build_stage4_plan
from causal_ssm_agent.flows.stages.stage4.agentic.stage4_repair.routing import classify_validation_outcome
from causal_ssm_agent.flows.stages.stage4.agentic.stage4_skeleton import derive_deterministic_spec
from causal_ssm_agent.flows.stages.stage4.assembly import validate_assembly
from causal_ssm_agent.flows.stages.stage4.model_spec_decisions import validate_model_spec_decisions_dict
from causal_ssm_agent.models.ssm_compiler import deserialize_ssm_spec

CHECKPOINT_PATH = REPO_ROOT / 'data/.private/GOLDEN/run/stage-4-checkpoints/review%3Aprior_system.pkl'
STAGE1B_PATH = REPO_ROOT / 'data/.private/GOLDEN/run/stage-1b.json'
STAGE3_PATH = REPO_ROOT / 'data/.private/GOLDEN/run/stage-3.json'
MODEL_DATA_PATH = REPO_ROOT / 'data/.private/GOLDEN/run/stage2-model-data.parquet'


def load_runtime():
    with CHECKPOINT_PATH.open('rb') as f:
        runtime = cloudpickle.load(f)
    causal_spec = json.loads(STAGE1B_PATH.read_text())['causal_spec']
    indicator_audits = json.loads(STAGE3_PATH.read_text())['indicators']
    data_for_model = pl.read_parquet(MODEL_DATA_PATH)
    return runtime, causal_spec, indicator_audits, data_for_model


def build_plan(causal_spec, skeleton):
    return build_stage4_plan(causal_spec, skeleton)


def summarize_validation(validation):
    if validation is None:
        return None
    failing_pp = [
        d.model_dump(mode='json')
        for d in validation.prior_predictive_diagnostics
        if not d.is_valid
    ]
    sensitivity_payload = validation.sensitivity_payload or {}
    weak = [
        {
            'index': direction.get('index'),
            'normalized_singular_value': direction.get('normalized_singular_value'),
            'top_loadings': direction.get('top_loadings', [])[:5],
        }
        for direction in sensitivity_payload.get('weak_directions', [])
        if isinstance(direction, dict) and direction.get('status') == 'fail'
    ][:5]
    return {
        'is_valid': validation.is_valid,
        'compile_ok': validation.compile_ok,
        'compile_error': validation.compile_error,
        'pp_checked': validation.pp_checked,
        'pp_valid': validation.pp_valid,
        'failing_prior_predictive_diagnostics': failing_pp,
        'sensitivity_consulted': validation.sensitivity_consulted,
        'sensitivity_supported': validation.sensitivity_supported,
        'sensitivity_valid': validation.sensitivity_valid,
        'sensitivity_deficiency_count': sensitivity_payload.get('deficiency_count'),
        'sensitivity_weak_directions_head': weak,
    }


def try_route(plan, runtime, validation):
    active_block = plan.get_block(runtime.domain.active_block_id)
    try:
        decision = classify_validation_outcome(plan, active_block, validation, runtime, feedback=None)
        payload = {'outcome': decision.outcome}
        if decision.repair_plan is not None:
            payload['scope_kind'] = decision.repair_plan.scope.scope_kind
            payload['scope_key'] = decision.repair_plan.scope.scope_key
            payload['scope_rank'] = decision.repair_plan.scope.scope_rank
            payload['block_ids'] = list(decision.repair_plan.block_ids)
            payload['reason'] = decision.repair_plan.scope.reason
        return payload
    except Exception as exc:
        return {'route_error': f'{type(exc).__name__}: {exc}'}


def pp_manifest_std_summary(validation, manifest_name):
    if validation is None or validation.compiled_ssm is None or not validation.pp_raw_samples:
        return None
    observations = validation.pp_raw_samples.get('observations')
    if observations is None:
        return None

    obs = np.asarray(observations)
    mask = validation.pp_raw_samples.get('observations_mask')
    mask = np.asarray(mask, dtype=bool) if mask is not None else None
    ssm = deserialize_ssm_spec(validation.compiled_ssm['spec'])
    manifest_names = list(ssm.manifest_names)
    manifest_idx = manifest_names.index(manifest_name)

    draw_stds = []
    for draw_idx in range(obs.shape[0]):
        values = obs[draw_idx, :, manifest_idx]
        if mask is not None and mask.shape == obs.shape:
            values = values[mask[draw_idx, :, manifest_idx]]
        values = values[np.isfinite(values)]
        if values.size >= 2:
            draw_stds.append(float(np.std(values)))

    audit = indicator_audits.get(manifest_name, {})
    profile = audit.get('profile', {}) if isinstance(audit, dict) else {}
    data_std = audit.get('std') if isinstance(audit, dict) else None
    if data_std is None:
        data_std = profile.get('std')
    return {
        'manifest_name': manifest_name,
        'median_implied_std': float(np.median(draw_stds)) if draw_stds else None,
        'min_implied_std': float(np.min(draw_stds)) if draw_stds else None,
        'max_implied_std': float(np.max(draw_stds)) if draw_stds else None,
        'n_draws_with_std': len(draw_stds),
        'data_std': data_std,
    }


def rebuild_locked_model_spec_from_checkpoint(base_model_spec, causal_spec):
    skeleton = derive_deterministic_spec(causal_spec)
    ambiguous_names = sorted({row['variable'] for row in skeleton.ambiguous_indicators})
    likelihood_by_var = {likelihood['variable']: likelihood for likelihood in base_model_spec['likelihoods']}
    decisions_payload = {
        'initialization_policy': base_model_spec.get('initialization_policy', 'stationary'),
        'observation_intercept_policy': base_model_spec.get('observation_intercept_policy', 'free'),
        'equilibrium_forcing': bool(base_model_spec.get('equilibrium_forcing', False)),
        'distribution_choices': [
            {
                'variable': name,
                'distribution': likelihood_by_var[name]['distribution'],
                'link': likelihood_by_var[name]['link'],
                'reasoning': likelihood_by_var[name].get('reasoning', 'replayed from checkpoint'),
            }
            for name in ambiguous_names
        ],
    }
    model_spec, errors = validate_model_spec_decisions_dict(
        decisions_payload,
        resolved_likelihoods=skeleton.resolved_likelihoods,
        ambiguous_indicators=skeleton.ambiguous_indicators,
        parameters=skeleton.all_params,
    )
    if errors or model_spec is None:
        raise ValueError(errors)
    return model_spec.model_dump(mode='json'), skeleton, decisions_payload


def filter_priors_for_model_spec(priors, model_spec):
    active_names = {parameter['name'] for parameter in model_spec['parameters']}
    filtered = {name: prior for name, prior in priors.items() if name in active_names}
    removed = sorted(set(priors) - set(filtered))
    return filtered, removed


runtime, causal_spec, indicator_audits, data_for_model = load_runtime()
base_model_spec = copy.deepcopy(runtime.domain.accepted.model_spec)
base_priors = copy.deepcopy(runtime.domain.accepted.authored_priors)
checkpoint_validation = runtime.domain.accepted.validation
print('Checkpoint path:', CHECKPOINT_PATH)
print('Active block:', runtime.domain.active_block_id)
print('Accepted priors:', len(base_priors))

/Users/ma9o/Desktop/causal-ssm-agent/trees/main/apps/data-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checkpoint path: /Users/ma9o/Desktop/causal-ssm-agent/trees/main/data/.private/GOLDEN/run/stage-4-checkpoints/review%3Aprior_system.pkl
Active block: review:prior_system
Accepted priors: 53


## Baseline: Raw Checkpoint Replay vs Checkpointed Accepted State

The checkpoint itself is paused on a compile-clean validation with one PPC failure on `monthly_eveningness_activity_timing`.

If I feed the raw checkpointed `model_spec` and `authored_priors` directly into today's compiler, compilation fails first because the raw payload still contains stale `obs_sd_*` authored priors for the two sleep-quality NB channels. That is a replay artifact, not the Stage 4 state the reducer was actually paused on.


In [2]:
checkpoint_summary = summarize_validation(checkpoint_validation)
checkpoint_scale = pp_manifest_std_summary(
    checkpoint_validation,
    'monthly_eveningness_activity_timing',
)
checkpoint_plan = build_plan(causal_spec, derive_deterministic_spec(causal_spec))
checkpoint_route = try_route(checkpoint_plan, runtime, checkpoint_validation)
raw_replay_validation = validate_assembly(
    base_model_spec,
    base_priors,
    data_for_model,
    indicator_audits,
    causal_spec,
)

print('CHECKPOINTED ACCEPTED VALIDATION')
pprint(checkpoint_summary)
print()
print('CHECKPOINTED ROUTING')
pprint(checkpoint_route)
print()
print('CHECKPOINTED MONTHLY_EVENINGNESS SCALE')
pprint(checkpoint_scale)
print()
print('RAW CHECKPOINT PAYLOAD REPLAY')
pprint(summarize_validation(raw_replay_validation))
if not raw_replay_validation.compile_ok:
    print()
    print('RAW REPLAY COMPILE ERROR')
    print(raw_replay_validation.compile_error)


CHECKPOINTED ACCEPTED VALIDATION
{'compile_error': None,
 'compile_ok': True,
 'failing_prior_predictive_diagnostics': [{'bad_manifest_names': [],
                                           'bad_sample_sites': [],
                                           'code': 'scale_mismatch',
                                           'compiled_flat_index': None,
                                           'compiled_site_name': None,
                                           'failing_draw_indices': [],
                                           'failure_stage': 'observation_sample',
                                           'first_bad_time_index': None,
                                           'is_valid': False,
                                           'issue': 'Scale mismatch for '
                                                    'monthly_eveningness_activity_timing: '
                                                    'implied std (0.01) vs '
                                           

## Normalization Fix: Rebuild the Locked `ModelSpec` Through Today's Stage 4 Path

The checked-in source does not expose a standalone `measurement_error_policy` field. What it does have is the current Stage 4 normalization path:

1. derive the deterministic skeleton from `causal_spec`,
2. replay the locked likelihood choices and model-level policies,
3. materialize the active parameter set for that locked model,
4. filter authored priors to those active parameter names.

Doing that removes the stale `obs_sd_*` priors for the two negative-binomial sleep-quality indicators and leaves the genuinely active `t0_*` chronotype surface intact. This makes the notebook compile cleanly again without modifying compiler source.


In [3]:
normalized_model_spec, skeleton, decisions_payload = rebuild_locked_model_spec_from_checkpoint(
    base_model_spec,
    causal_spec,
)
normalized_priors, removed_prior_names = filter_priors_for_model_spec(base_priors, normalized_model_spec)
normalized_validation = validate_assembly(
    normalized_model_spec,
    normalized_priors,
    data_for_model,
    indicator_audits,
    causal_spec,
)
normalized_plan = build_plan(causal_spec, skeleton)
normalized_runtime = copy.deepcopy(runtime)
normalized_runtime.domain.accepted.model_spec = copy.deepcopy(normalized_model_spec)
normalized_runtime.domain.accepted.authored_priors = copy.deepcopy(normalized_priors)
normalized_runtime.domain.accepted.validation = normalized_validation
normalized_route = try_route(normalized_plan, normalized_runtime, normalized_validation)

print('REPLAYED MODEL DECISIONS')
pprint(decisions_payload)
print()
print('NORMALIZED ACTIVE PARAMETER COUNT', len(normalized_model_spec['parameters']))
print('REMOVED STALE PRIORS', removed_prior_names)
print()
print('NORMALIZED REPLAY VALIDATION')
pprint(summarize_validation(normalized_validation))
print()
print('NORMALIZED REPLAY ROUTING')
pprint(normalized_route)


REPLAYED MODEL DECISIONS
{'distribution_choices': [{'distribution': 'negative_binomial',
                           'link': 'log',
                           'reasoning': 'Count data with significant '
                                        'overdispersion (var/mean = 8.28 >> 1) '
                                        'and high zero fraction (93.5%). '
                                        'Negative binomial handles '
                                        'overdispersed counts better than '
                                        'Poisson. Log link is the natural '
                                        'choice for count data.',
                           'variable': 'anxiety_depression_related_search_count'},
                          {'distribution': 'negative_binomial',
                           'link': 'log',
                           'reasoning': 'Count data with significant '
                                        'overdispersion (var/mean = 4.98). '
                  

## Status After the Normalization Fix

The compile failure was a notebook replay issue, not a compiler bug in the current checked-in source.

Once the locked `ModelSpec` is rebuilt through today's Stage 4 path, the notebook lands on the same substantive blocker as the checkpointed accepted validation:
- compile is clean,
- `obs_sd_*` is no longer part of the active prior surface for the NB sleep-quality indicators,
- PPC still fails on `monthly_eveningness_activity_timing`,
- the reducer still cannot derive a concrete structural repair scope for that PPC failure.
